In [1]:
import json
import os
from datetime import date
from dotenv import load_dotenv
from llama_stack_client import LlamaStackClient
from time import sleep

load_dotenv()

LLAMA_STACK_URL = os.getenv("LLAMA_STACK_URL", "http://127.0.0.1:8321")
MODEL_ID = "vllm/openai/gpt-oss-120b"
MODEL_LABEL = "GPT-OSS-120b"
VLLM_VERSION = "0.11.2+rhai5"
RUN_DATE = os.getenv("RUN_DATE", date.today().isoformat())

MAX_TOOL_CALLS = 15
MAX_INFER_ITERS = 20

# Use llama-stack client (supports max_infer_iters parameter, unlike OpenAI client)
client = LlamaStackClient(base_url=LLAMA_STACK_URL)

# Note: Using llama-stack client to pass max_infer_iters directly (not supported by OpenAI client).
# max_infer_iters controls the total number of iterations (tool calls + final message).
# max_tool_calls limits the number of tool calls within those iterations.

PROMPTS = [
    "Use the file search tool to find information in the uploaded documents. What information is available?",
    "Search the uploaded documents and summarize the key points.",
    "What are the main topics covered in the uploaded documents? Use file search to find this information."
]

# Vector store setup - automatically create if needed
VECTOR_STORE_IDS = os.getenv("VECTOR_STORE_IDS", "").split(",") if os.getenv("VECTOR_STORE_IDS") else []
VECTOR_STORE_NAME = "file-search-reference-docs"
REFERENCE_FILE = "reference.txt"

# Create vector store and upload file if not already set
if not VECTOR_STORE_IDS or not VECTOR_STORE_IDS[0]:
    print("Setting up vector store...")
    
    # Check if reference file exists
    if not os.path.exists(REFERENCE_FILE):
        raise FileNotFoundError(f"Reference file '{REFERENCE_FILE}' not found. Please create it in the same directory as this notebook.")
    
    # Upload the file
    print(f"Uploading {REFERENCE_FILE}...")
    with open(REFERENCE_FILE, "rb") as f:
        file = client.files.create(file=f, purpose="assistants")
    print(f"✅ Uploaded file: {file.id}")
    
    # Create vector store
    print("Creating vector store...")
    vector_store = client.vector_stores.create(name=VECTOR_STORE_NAME)
    print(f"✅ Created vector store: {vector_store.id}")
    
    # Add file to vector store
    print("Adding file to vector store...")
    vector_store_file = client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=file.id
    )
    print(f"✅ Added file to vector store")
    
    # Wait for file processing
    print("Waiting for file processing...")
    max_wait = 60  # seconds
    waited = 0
    while waited < max_wait:
        sleep(2)
        waited += 2
        file_status = client.vector_stores.files.retrieve(
            vector_store_id=vector_store.id,
            file_id=file.id
        )
        if file_status.status == "completed":
            print(f"✅ File processed successfully after {waited}s")
            break
        elif file_status.status == "failed":
            print(f"❌ File processing failed after {waited}s")
            break
        if waited % 10 == 0:
            print(f"  Still processing... ({waited}s)")
    
    if file_status.status != "completed":
        print(f"⚠️  Warning: File status is {file_status.status} after {waited}s")
    
    VECTOR_STORE_IDS = [vector_store.id]
    print(f"\n✅ Vector store setup complete. ID: {VECTOR_STORE_IDS[0]}")
    print(f"   You can set VECTOR_STORE_IDS={VECTOR_STORE_IDS[0]} as an environment variable to reuse it.")
else:
    # Verify vector store exists
    try:
        vs = client.vector_stores.retrieve(vector_store_id=VECTOR_STORE_IDS[0])
        print(f"✅ Using existing vector store: {VECTOR_STORE_IDS[0]}")
        print(f"   Status: {vs.status}, Files: {vs.file_counts.total}")
    except Exception as e:
        print(f"❌ Error accessing vector store {VECTOR_STORE_IDS[0]}: {e}")
        print("   Please check the VECTOR_STORE_IDS or let the script create a new one.")
        raise

def run_prompt(prompt):
    print(f"\nUser: {prompt}")
    response = client.responses.create(
        model=MODEL_ID,
        tools=[{"type": "file_search", "vector_store_ids": VECTOR_STORE_IDS}],
        input=prompt,
        max_tool_calls=MAX_TOOL_CALLS,
        max_infer_iters=MAX_INFER_ITERS
    )

    print(f"Response ID: {response.id}")
    print(f"Response status: {response.status}")
    
    file_search_triggered = False
    tool_call_count = 0
    message_count = 0

    for item in response.output:
        item_type = getattr(item, "type", "unknown")
        item_status = getattr(item, "status", "unknown")
        print(f"  - Type: {item_type}, Status: {item_status}")
        if "file_search" in item_type:
            file_search_triggered = True
            tool_call_count += 1
        elif item_type == "message":
            message_count += 1

    print("\nDiagnostics:")
    print(f"  File search triggered: {file_search_triggered}")
    print(f"  Tool calls: {tool_call_count}")
    print(f"  Messages: {message_count}")
    
    print(f"  Total iterations: {tool_call_count + message_count}")
    
    # Check for max_infer_iters issue
    if response.status == "incomplete" and tool_call_count >= MAX_TOOL_CALLS and message_count == 0:
        print(f"  ⚠️  WARNING: Likely hit max_tool_calls limit ({MAX_TOOL_CALLS})")
        print(f"     Model used {tool_call_count} tool calls, leaving no iterations for final message")
    elif response.status == "incomplete" and (tool_call_count + message_count) >= MAX_INFER_ITERS:
        print(f"  ⚠️  WARNING: Likely hit max_infer_iters limit ({MAX_INFER_ITERS})")
        print(f"     Total iterations: {tool_call_count + message_count}")
    
    print("\n" + "=" * 60)
    print(f"Response from {MODEL_LABEL} with vLLM {VLLM_VERSION} ({RUN_DATE})")
    print("=" * 60)
    if response.output_text:
        print(response.output_text)
    else:
        print("(empty response)")
    return response


# Store all responses for JSON metadata
all_responses = []

try:
    for prompt in PROMPTS:
        response = run_prompt(prompt)
        all_responses.append(response)
        sleep(10)  # Brief pause between prompts
finally:
    # llama-stack client doesn't need explicit close, but keeping for consistency
    pass

# Generate summary and JSON, then update markdown cells
import json as json_lib

summary_md = "# Run Output Summary\n\n"
for i, (prompt, response) in enumerate(zip(PROMPTS, all_responses), 1):
    file_search_triggered = any("file_search" in str(item.type) for item in response.output)
    tool_call_count = sum(1 for item in response.output if "file_search" in str(item.type))
    message_count = sum(1 for item in response.output if item.type == "message")
    
    summary_md += f"## Prompt {i}\n\n"
    summary_md += f"**Status:** {response.status} | **File search:** {file_search_triggered} | **Tool calls:** {tool_call_count} | **Messages:** {message_count}\n\n"
    output_text = response.output_text or "(empty response)"
    summary_md += f"**Response:** {output_text[:200]}{'...' if len(output_text) > 200 else ''}\n\n"
    summary_md += "---\n\n"

json_md = "# Full Response JSON (with metadata)\n\n"
for i, (prompt, response) in enumerate(zip(PROMPTS, all_responses), 1):
    json_md += f"## Prompt {i}\n\n"
    json_md += f"**Prompt:** {prompt}\n\n"
    json_md += "```json\n"
    json_md += json_lib.dumps(response.model_dump(), indent=2, default=str)
    json_md += "\n```\n\n"

# Update notebook cells directly
import nbformat
from nbformat import v4 as nbf

nb_path = "GPT-OSS-120b_with_vLLM_0.11.2+rhai5.ipynb"
with open(nb_path, "r") as f:
    nb = nbformat.read(f, as_version=4)

# Update summary cell (cell index 1)
if len(nb.cells) > 1:
    nb.cells[1].source = summary_md

# Update JSON cell (cell index 2)  
if len(nb.cells) > 2:
    nb.cells[2].source = json_md

with open(nb_path, "w") as f:
    nbformat.write(nb, f)

print("✅ Summary and JSON data added to notebook")


INFO:httpx:HTTP Request: POST http://127.0.0.1:8321/v1/files "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://127.0.0.1:8321/v1/vector_stores "HTTP/1.1 200 OK"


Setting up vector store...
Uploading reference.txt...
✅ Uploaded file: file-f343be92da044c09abe0cf1d8ae641f8
Creating vector store...
✅ Created vector store: vs_960fc02f-4134-4382-955c-bdadaf305ad6
Adding file to vector store...


INFO:httpx:HTTP Request: POST http://127.0.0.1:8321/v1/vector_stores/vs_960fc02f-4134-4382-955c-bdadaf305ad6/files "HTTP/1.1 200 OK"


✅ Added file to vector store
Waiting for file processing...


...

Response ID: resp_f29683e8-56ad-4876-9293-716025fea4e3
Response status: completed
  - Type: file_search_call, Status: completed
  - Type: message, Status: completed

Diagnostics:
  File search triggered: True
  Tool calls: 1
  Messages: 1
  Total iterations: 2

Response from GPT-OSS-120b with vLLM 0.11.2+rhai5 (2026-02-02)
**Summary of the uploaded document**

- **Hamid Moghani** is a Red Hat Engineer who works on the **Agentic and MCP team**.  
- He is currently **manually testing Llama Stack tools with GPT‑OSS models on VLLM**.  
- His **manager is Justin**, and his teammates include **Bill, Jaideep, Emilio, Akanksha, Khaled, Mike, Shabana, Gordon, Nehanth, and Robin**.  
- **Mike and Hamid are planning to build a test strategy and a test‑automation framework for the Llama‑Stack responses API**.  

These points are drawn from the reference document provided.



User: What are the main topics covered in the uploaded documents? Use file search to find this information.


INFO:httpx:HTTP Request: POST http://127.0.0.1:8321/v1/responses "HTTP/1.1 200 OK"


Response ID: resp_8fe8d18f-caa6-4b32-941b-acd633661582
Response status: completed
  - Type: file_search_call, Status: completed
  - Type: message, Status: completed

Diagnostics:
  File search triggered: True
  Tool calls: 1
  Messages: 1
  Total iterations: 2

Response from GPT-OSS-120b with vLLM 0.11.2+rhai5 (2026-02-02)
The uploaded document mainly covers a professional profile and current project work. It describes Hamid Moghani, a Red Hat engineer on the Agentic and MCP team, and lists his manager (Justin) and teammates (Bill, Jaideep, Emilio, Akanksha, Khaled, Mike, Shabana, Gordon, Nehanth, Robin). It details his current activities, which include manually testing the Llama Stack tools with GPT‑OSS models on VLLM, and outlines plans with Mike to develop a test‑strategy and test‑automation framework for the Llama‑Stack responses API .


✅ Summary and JSON data added to notebook


# Run Output Summary

## Prompt 1

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** The uploaded document contains a short profile of a Red Hat engineer:

- **Name:** Hamid Moghani  
- **Team:** Agentic and MCP  
- **Current work:** Manually testing Llama Stack tools with GPT‑OSS mod...

---

## Prompt 2

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** **Summary of the uploaded document**

- Hamid Moghani is a Red Hat Engineer who works in the **Agentic and MCP team**.  
- He is currently **manually testing Llama Stack tools** with GPT‑OSS models on...

---

## Prompt 3

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** The uploaded document primarily discusses:

- **Hamid Moghani’s role** – a Red Hat Engineer in the Agentic and MCP team  
- **Current project focus** – manually testing Llama Stack tools with GPT‑OSS ...

---



# Full Response JSON (with metadata)

## Prompt 1

**Prompt:** Use the file search tool to find information in the uploaded documents. What information is available?

```json
{
  "id": "resp_2cc3fcc5-bf6f-4168-846b-c387ab5e2942",
  "created_at": 1770071605,
  "model": "vllm/openai/gpt-oss-120b",
  "output": [
    {
      "id": "fc_5db0f5eb-a9d8-4a80-9324-b9ccd010797a",
      "queries": [
        "uploaded documents"
      ],
      "status": "completed",
      "results": [
        {
          "attributes": {
            "filename": "reference.txt",
            "chunk_id": "e403e0a2-0c4c-ad17-7e51-52e2390f14f9",
            "document_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
            "token_count": 102.0,
            "metadata_token_count": 7.0
          },
          "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
          "filename": "file-2842f3772a7c4deea2abdcc9111d9f12",
          "score": 0.486054131427206,
          "text": "Hamid Moghani is a Red Hat Engineer who works in Agentic and MCP team. \nHe is currently working on manually testing llama stack tools with GPT-OSS models on VLLM.\nHis manager is Justin and and teammates are Bill, Jaideep, Emilio, Akanksha, Khaled, Mike,  Shabana, Gordon, Nehanth, and Robin.\nMike and Hamid are planning to build a Test Staretgy and Test Automation framework for llama-stack responses API.\n"
        }
      ],
      "type": "file_search_call"
    },
    {
      "content": [
        {
          "text": "The uploaded document contains a short profile of a Red\u202fHat engineer:\n\n- **Name:** Hamid\u202fMoghani  \n- **Team:** Agentic and MCP  \n- **Current work:** Manually testing Llama\u202fStack tools with GPT\u2011OSS models on VLLM  \n- **Manager:** Justin  \n- **Teammates:** Bill, Jaideep, Emilio, Akanksha, Khaled, Mike, Shabana, Gordon, Nehanth, and Robin  \n- **Upcoming project:** Hamid and Mike plan to build a test strategy and test\u2011automation framework for Llama\u2011Stack responses API",
          "annotations": [
            {
              "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
              "filename": "reference.txt",
              "index": 468,
              "type": "file_citation"
            }
          ],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_50d892c8-f5e0-402a-8347-878be8a1df39",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 15,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_b4880390-d29d-4629-b584-080e01e811b4"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 710,
    "output_tokens": 306,
    "total_tokens": 1016,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770071607,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

## Prompt 2

**Prompt:** Search the uploaded documents and summarize the key points.

```json
{
  "id": "resp_48b3970e-fbb8-4a5c-afd0-d7a53403dc67",
  "created_at": 1770071607,
  "model": "vllm/openai/gpt-oss-120b",
  "output": [
    {
      "id": "fc_c486d2ce-077e-4603-b841-631d6e2f6088",
      "queries": [
        "uploaded documents"
      ],
      "status": "completed",
      "results": [
        {
          "attributes": {
            "filename": "reference.txt",
            "chunk_id": "e403e0a2-0c4c-ad17-7e51-52e2390f14f9",
            "document_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
            "token_count": 102.0,
            "metadata_token_count": 7.0
          },
          "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
          "filename": "file-2842f3772a7c4deea2abdcc9111d9f12",
          "score": 0.486054131427206,
          "text": "Hamid Moghani is a Red Hat Engineer who works in Agentic and MCP team. \nHe is currently working on manually testing llama stack tools with GPT-OSS models on VLLM.\nHis manager is Justin and and teammates are Bill, Jaideep, Emilio, Akanksha, Khaled, Mike,  Shabana, Gordon, Nehanth, and Robin.\nMike and Hamid are planning to build a Test Staretgy and Test Automation framework for llama-stack responses API.\n"
        }
      ],
      "type": "file_search_call"
    },
    {
      "content": [
        {
          "text": "**Summary of the uploaded document**\n\n- Hamid Moghani is a Red\u202fHat Engineer who works in the **Agentic and MCP team**.  \n- He is currently **manually testing Llama\u202fStack tools** with GPT\u2011OSS models on VLLM.  \n- His **manager** is **Justin**.  \n- His **teammates** include **Bill, Jaideep, Emilio, Akanksha, Khaled, Mike, Shabana, Gordon, Nehanth, and Robin**.  \n- **Mike and Hamid** are planning to **build a Test Strategy and Test Automation framework** for the Llama\u2011Stack **responses API**.",
          "annotations": [
            {
              "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
              "filename": "reference.txt",
              "index": 493,
              "type": "file_citation"
            }
          ],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_e8e57196-b042-419a-87a9-239c5799745c",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 15,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_b4880390-d29d-4629-b584-080e01e811b4"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 694,
    "output_tokens": 320,
    "total_tokens": 1014,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770071610,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

## Prompt 3

**Prompt:** What are the main topics covered in the uploaded documents? Use file search to find this information.

```json
{
  "id": "resp_93b43bc9-752d-4441-af58-ca6e80d8c8c2",
  "created_at": 1770071610,
  "model": "vllm/openai/gpt-oss-120b",
  "output": [
    {
      "id": "fc_40ecec5c-62f9-4ca0-8102-35de95792699",
      "queries": [
        "topics covered in the uploaded documents"
      ],
      "status": "completed",
      "results": [
        {
          "attributes": {
            "filename": "reference.txt",
            "chunk_id": "e403e0a2-0c4c-ad17-7e51-52e2390f14f9",
            "document_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
            "token_count": 102.0,
            "metadata_token_count": 7.0
          },
          "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
          "filename": "file-2842f3772a7c4deea2abdcc9111d9f12",
          "score": 0.5183777484990968,
          "text": "Hamid Moghani is a Red Hat Engineer who works in Agentic and MCP team. \nHe is currently working on manually testing llama stack tools with GPT-OSS models on VLLM.\nHis manager is Justin and and teammates are Bill, Jaideep, Emilio, Akanksha, Khaled, Mike,  Shabana, Gordon, Nehanth, and Robin.\nMike and Hamid are planning to build a Test Staretgy and Test Automation framework for llama-stack responses API.\n"
        }
      ],
      "type": "file_search_call"
    },
    {
      "content": [
        {
          "text": "The uploaded document primarily discusses:\n\n- **Hamid\u202fMoghani\u2019s role** \u2013 a Red\u202fHat Engineer in the Agentic and MCP team  \n- **Current project focus** \u2013 manually testing Llama\u202fStack tools with GPT\u2011OSS models on VLLM  \n- **Team structure** \u2013 manager (Justin) and teammates (Bill, Jaideep, Emilio, Akanksha, Khaled, Mike, Shabana, Gordon, Nehanth, Robin)  \n- **Planned work** \u2013 developing a test strategy and an automated test\u2011automation framework for the Llama\u2011Stack\u202fResponses API",
          "annotations": [
            {
              "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
              "filename": "reference.txt",
              "index": 119,
              "type": "file_citation"
            },
            {
              "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
              "filename": "reference.txt",
              "index": 214,
              "type": "file_citation"
            },
            {
              "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
              "filename": "reference.txt",
              "index": 351,
              "type": "file_citation"
            },
            {
              "file_id": "file-2842f3772a7c4deea2abdcc9111d9f12",
              "filename": "reference.txt",
              "index": 478,
              "type": "file_citation"
            }
          ],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_e411b441-c907-4582-abda-65bdf39b1fe6",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 15,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_b4880390-d29d-4629-b584-080e01e811b4"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 721,
    "output_tokens": 519,
    "total_tokens": 1240,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770071614,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

